# Sohn et al. (2019) on dandiset 000130: single-trial GPFA analyses

Paper: Sohn, Narain, Meirhaeghe and Jazayeri (2019), Neuron 103:934, doi:10.1016/j.neuron.2019.06.012;
claims file `claims/sohn-2019.yaml` (from the author manuscript, PMC6805134). Dataset: DANDI:000130,
version 0.241017.1448, the version this attempt was asked to use (the most recent published one;
the paper cites none), streamed from the archive. The trial-averaged claims are in
`reproduce_sohn_2019.ipynb`; this notebook tests the two GPFA claims, gpfa-variance-inverted-u and
gpfa-mean-sigmoid (Figure 6C and 6D). The paper ran this analysis on "high-yield sessions (N=48 for
monkey H)"; this session has 48 units that pass the unit selection.

GPFA is elephant's implementation of Yu et al. (2009). It is fit per dataset (effector and direction)
on single trials from Ready to Set, both priors, in 20 ms bins, with the latent dimensionality chosen
from 4 and 8 by 3-fold cross-validated log-likelihood, as the criterion of gpfa-variance-inverted-u
gives. Fitting takes about 4 minutes per dataset.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
import time, resource, warnings, logging
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from scipy import stats
import rsg

warnings.filterwarnings("ignore")
logging.getLogger("elephant").setLevel(logging.ERROR)
T0 = time.time()
DATASET = f"dandi:{rsg.DANDISET}/{rsg.VERSION}"
CLAIMS = {c["id"]: c for c in yaml.safe_load(open("../../claims/sohn-2019.yaml"))["claims"]}
pd.set_option("display.width", 200)


def quote(cid):
    c = CLAIMS[cid]
    print(f"[{cid}] {c['statement']}\nCriterion: {c.get('criterion', '(none)')}")

In [ ]:
sess = rsg.load_session(rsg.TRAIN_PATH)
good = rsg.good_trials(sess["trials"])
ui = rsg.select_units(sess, good)
spikes = [sess["spikes"][i] for i in ui]
print(DATASET, rsg.TRAIN_PATH)
print("trials analysed (completed, not outlier):", len(good), "| units selected:", len(ui), "of", len(sess["units"]))
print(pd.crosstab([good.dataset, good.prior], good.ts))
print("Ready to Set (ms) from the trial table: %.1f to %.1f; ts values %s" % (
    ((good.set_time - good.ready_time) * 1000).min(), ((good.set_time - good.ready_time) * 1000).max(), sorted(good.ts.unique())))
print("spikes per trial from Ready to the ts-th ms: median %.0f" % np.median(
    [sum(np.searchsorted(s, r + t / 1000) - np.searchsorted(s, r) for s in spikes) for r, t in zip(good.ready_time[:50], good.ts[:50])]))

## GPFA fits

Each trial runs from Ready to Ready + ts (the bins end on the ts grid, every ts being a multiple of
20 ms). The fold assignment is seeded.

In [ ]:
FITS = {}
for j, ds in enumerate(sorted(good.dataset.unique())):
    t = time.time()
    d = good[good.dataset == ds].reset_index(drop=True)
    best, cv, lat = rsg.gpfa_fit(spikes, d, x_dims=(4, 8), n_folds=3, seed=100 + j)
    lengths_ok = all(x.shape[0] == ts // rsg.BIN for x, ts in zip(lat, d.ts.values.astype(int)))
    FITS[ds] = dict(d=d, dim=best, cv=cv, lat=lat)
    print(f"{ds}: {len(d)} trials; CV log-likelihood {({k: round(v, 1) for k, v in cv.items()})}; chosen dimensionality {best}; "
          f"latent lengths match ts: {lengths_ok}; {time.time() - t:.0f} s")

## gpfa-variance-inverted-u

In [ ]:
quote("gpfa-variance-inverted-u")


def projections(F, zscore="prior"):
    """Xu at Set per prior. zscore="prior" z-scores Xu over the prior's trials (claims file);
    zscore="dataset" over all trials of the dataset, both priors (the Figure 6C legend: "We z-scored
    Xu of all trials before computing the variance for each ts")."""
    rows, proj = [], {}
    d, lat = F["d"], F["lat"]
    xu_all = {}
    for prior in ["Short", "Long"]:
        tsv = rsg.PRIOR_TS[prior]
        idx = np.flatnonzero(d.prior.values == prior)
        nb = tsv[-1] // rsg.BIN
        L = np.full((len(idx), nb, lat[0].shape[1]), np.nan)
        for r, i in enumerate(idx):
            L[r, :lat[i].shape[0]] = lat[i]
        avg = np.nanmean(L, axis=0)  # attrition: each bin averages the trials still before Set
        ends = np.arange(rsg.BIN, tsv[-1] + 1, rsg.BIN)
        u = rsg.encoding_axis(avg, ends, tsv[0], tsv[-1])
        xu = np.array([lat[i][-1] @ u for i in idx])  # state in the last bin before Set
        xu_all[prior] = (xu, d.ts.values[idx])
    pooled = np.concatenate([v[0] for v in xu_all.values()])
    for prior in ["Short", "Long"]:
        tsv = rsg.PRIOR_TS[prior]
        xu, ts_i = xu_all[prior]
        ref = xu if zscore == "prior" else pooled
        z = (xu - ref.mean()) / ref.std(ddof=1)
        var = np.array([z[ts_i == x].var(ddof=1) for x in tsv])
        mean = np.array([xu[ts_i == x].mean() for x in tsv])
        proj[prior] = dict(xu=xu, ts=ts_i, var=var, mean=mean)
        rows.append(dict(prior=prior, dim=lat[0].shape[1], n=[int((ts_i == x).sum()) for x in tsv],
                         var=np.round(var, 3).tolist(), quad_coef_per_s2=np.polyfit(tsv / 1000.0, var, 2)[0],
                         extremes_mean_var=(var[0] + var[-1]) / 2, middle3_mean_var=var[1:4].mean()))
    return rows, proj


rows, PROJ = [], {}
for ds, F in FITS.items():
    r, p = projections(F)
    rows += [dict(dataset=ds, **x) for x in r]
    PROJ.update({(ds, k): v for k, v in p.items()})
VAR = pd.DataFrame(rows)
print(VAR.to_string(index=False))
w = stats.wilcoxon(VAR.quad_coef_per_s2, alternative="less")
print(f"\nquadratic coefficient below zero in {int((VAR.quad_coef_per_s2 < 0).sum())} of 8; one-tailed signed-rank (coefficient < 0) p = {w.pvalue:.4f}")
print(f"variance at the extremes below the mean of the middle three ts in {int((VAR.extremes_mean_var < VAR.middle3_mean_var).sum())} of 8")
allv = np.vstack(VAR["var"].values)
print("mean variance profile over the 8 dataset-priors (by ts rank):", np.round(allv.mean(0), 3).tolist())

In [ ]:
# variant: Xu z-scored over all trials of the dataset (both priors), the Figure 6C legend's wording
rows_z = []
for ds, F in FITS.items():
    r, _ = projections(F, zscore="dataset")
    rows_z += [dict(dataset=ds, **x) for x in r]
VARZ = pd.DataFrame(rows_z)
print(VARZ[["dataset", "prior", "var", "quad_coef_per_s2"]].to_string(index=False))
wz = stats.wilcoxon(VARZ.quad_coef_per_s2, alternative="less")
print(f"z-scored over both priors: quadratic coefficient below zero in {int((VARZ.quad_coef_per_s2 < 0).sum())} of 8; "
      f"one-tailed signed-rank p = {wz.pvalue:.4f}")


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
for (ds, prior), p in PROJ.items():
    ax[0 if prior == "Short" else 1].plot(rsg.PRIOR_TS[prior], p["var"], "o-", lw=0.8, label=ds)
for a, prior in zip(ax, ["Short", "Long"]):
    a.set_title(f"{prior}: variance of z-scored Xu")
    a.set_xlabel("ts (ms)")
ax[0].legend(fontsize=7)
plt.tight_layout()
plt.show()
print("plotted variances range %.3f to %.3f (all shown)" % (allv.min(), allv.max()))

Variants. elephant fits GPFA on 20-bin segments of the trials, with overlaps drawn at random, so the
fit depends on the seed. The same analysis is run with 8 latent dimensions (the cross-validated
choice) refit under three other seeds, and with 4 latent dimensions, the other value the criterion
allows. Each is shown with Xu z-scored over the prior's trials (claims file) and over all trials of the
dataset (the Figure 6C legend: "We z-scored Xu of all trials"), a reading asked of a reader.

In [ ]:
variants = [("8 dims, seed 1", 8, 1), ("8 dims, seed 2", 8, 2), ("8 dims, seed 3", 8, 3), ("4 dims, seed 0", 4, 0)]
vrows = []
for name, dim, seed in variants:
    t = time.time()
    rows_v = []
    for j, (ds, F) in enumerate(FITS.items()):
        _, _, lat_v = rsg.gpfa_fit(spikes, F["d"], x_dims=(dim,), seed=1000 * seed + 100 + j)
        for zs in ["prior", "dataset"]:
            r, _ = projections(dict(d=F["d"], lat=lat_v), zscore=zs)
            rows_v += [dict(dataset=ds, zscore=zs, **x) for x in r]
    for zs, V in pd.DataFrame(rows_v).groupby("zscore", sort=False):
        vrows.append(dict(variant=name, zscore=zs, n_negative=int((V.quad_coef_per_s2 < 0).sum()),
                          p_one_tailed=stats.wilcoxon(V.quad_coef_per_s2, alternative="less").pvalue,
                          quad_coefs=np.round(V.quad_coef_per_s2.values, 2).tolist(), seconds=round(time.time() - t)))
VARV = pd.DataFrame(vrows)
print(VARV.to_string(index=False))
print("main fit:", int((VAR.quad_coef_per_s2 < 0).sum()), "negative, p =", round(w.pvalue, 4),
      "| coefficients", np.round(VAR.quad_coef_per_s2.values, 2).tolist())

**Result: not reproduced.** With the 8 latent dimensions that cross-validation chose in all four
datasets, the quadratic coefficient is negative in 6 of the 8 dataset-priors, and the one-tailed
signed-rank test gives p = 0.055, just above 0.05. The result is close to the threshold: refitting
under three other seeds gives p = 0.055, 0.039 and 0.055, and 4 latent dimensions give p = 0.098, with
6 of 8 coefficients negative every time. With Xu z-scored over all trials of the dataset, the reading of
the Figure 6C legend, p is 0.074 for the main fit and 0.039 to 0.074 for the variants, again 6 of 8
negative, so this reading does not change the outcome. The mean variance profile over the 8
dataset-priors (0.78, 0.91, 0.85, 0.51, 0.45 by ts rank) falls toward the long end more than it dips at
the short end. The variances rest on 14 to 38 trials per ts.

## gpfa-mean-sigmoid

In [ ]:
quote("gpfa-mean-sigmoid")
rows = []
for (ds, prior), p in PROJ.items():
    tsv = rsg.PRIOR_TS[prior].astype(float)
    inc = np.diff(p["mean"])
    rows.append(dict(dataset=ds, prior=prior, mean_xu=np.round(p["mean"], 3).tolist(), increments=np.round(inc, 3).tolist(),
                     extreme=(inc[0] + inc[3]) / 2, middle=(inc[1] + inc[2]) / 2, slope=stats.linregress(tsv, p["mean"]).slope))
MS = pd.DataFrame(rows)
print(MS.to_string(index=False))
w = stats.wilcoxon(MS.middle, MS.extreme)
print(f"\nmiddle increment above extreme in {int((MS.middle > MS.extreme).sum())} of 8; two-sided signed-rank p = {w.pvalue:.4f}")
sl = MS.pivot(index="dataset", columns="prior", values="slope")
sl["Short_higher"] = sl.Short > sl.Long
print(sl.to_string())
print(f"Short slope above Long in {int(sl.Short_higher.sum())} of 4; one-tailed signed-rank p = {stats.wilcoxon(sl.Short, sl.Long, alternative='greater').pvalue:.4f}")

**Result: reproduced.** The middle increment of the mean projection exceeds the extreme one in 7 of
8 dataset-priors (two-sided signed-rank p = 0.0156), and the slope of the mean projection on ts is
higher for Short than for Long in all 4 datasets (one-tailed p = 0.0625, the paper's value).

## Summary

| claim | outcome |
|---|---|
| gpfa-variance-inverted-u | not reproduced: quadratic coefficient negative in 6 of 8, one-tailed p = 0.055 (0.039 to 0.098 in the variants; 0.074 with Xu z-scored over both priors) |
| gpfa-mean-sigmoid | reproduced: middle above extreme increments in 7 of 8 (p = 0.0156); Short slope higher in 4 of 4 |

The trial-averaged claims and the summary of the whole attempt are in `reproduce_sohn_2019.ipynb`.
The paper fit GPFA in MATLAB (Yu et al. 2009; Cowley et al. 2013); elephant's port is used here, with
the dimensionality rule and the 3-fold cross-validation of the claims file. The paper used one
high-yield session of 48 units for monkey H; this session also has 48 selected units, but whether it
is the same session cannot be told from the files.

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and
the differences from the paper's method.

In [ ]:
print(f"runtime {time.time() - T0:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")